In [2]:

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.model_selection import KFold
import sklearn.linear_model as lm

from utils import Preprocessor, compute_error

seed = 42
np.random.seed(seed)

sns.set_style('darkgrid')
sns.set_theme(font_scale=1.5)


df = pd.read_csv("data/heartDisease.csv")
df = df.drop(labels=["row.names"], axis=1)


In [6]:
classification_results = pd.read_csv("tlcv_results/classification_results.csv")
classification_results_grouped = classification_results.groupby(["model", "param_name"], dropna=False)["param_val"].agg(pd.Series.mode).to_frame().reset_index()

clf_logistic_lam_parameter = classification_results_grouped[(classification_results_grouped["model"] == "LogisticRegression") & (classification_results_grouped["param_name"] == "lambda")]["param_val"].values[0]
if np.array(clf_logistic_lam_parameter).shape != ():
    clf_logistic_lam_parameter = clf_logistic_lam_parameter[0]
clf_logistic_lam_parameter = float(clf_logistic_lam_parameter)
clf_logistic_lam_parameter

0.0286606761694825

In [7]:

best_lm = lm.LogisticRegression(penalty="l2", C=1/clf_logistic_lam_parameter)

preprocessor = Preprocessor(task='classification')

X_preprocessed, y = preprocessor.fit_transform(df)

best_lm.fit(X_preprocessed, y)


for coef, feature_name in zip(best_lm.coef_.reshape(-1), preprocessor.get_feature_names_out().tolist()):
    print(feature_name, "\t\t", coef)





num__obesity 		 -0.1601633702403585
num__alcohol 		 -0.05197875081368785
num__typea 		 0.3745036130192703
num__tobacco 		 0.41375471227848903
num__sbp 		 0.15743714105083245
num__age 		 0.7470796178314855
num__ldl 		 0.33819196520069067
cat__famhist 		 0.890467407489024


/Users/krusand/Documents/GitHub/02452-ML-Project/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [10]:
coef_obesity = -0.1601633702403585
coef_alcohol = -0.05197875081368785
coef_typea = 0.3745036130192703
coef_tobacco = 0.41375471227848903
coef_sbp = 0.15743714105083245
coef_age = 0.7470796178314855
coef_ldl = 0.33819196520069067
coef_famhist = 0.890467407489024

In [11]:
row_sample = X_preprocessed.iloc[1:2]
row_sample

,num__obesity,num__alcohol,num__typea,num__tobacco,num__sbp,num__age,num__ldl,cat__famhist
1,0.671373,0.273101,0.193344,-0.491199,0.277089,1.383115,-0.15968,0.0


In [12]:
best_lm.intercept_

array([-1.28544184])

In [16]:
pred_val = (best_lm.intercept_
 + coef_obesity * row_sample["num__obesity"].values 
 + coef_alcohol * row_sample["num__alcohol"].values 
 + coef_typea * row_sample["num__typea"].values 
 + coef_tobacco * row_sample["num__tobacco"].values 
 + coef_sbp * row_sample["num__sbp"].values 
 + coef_age * row_sample["num__age"].values 
 + coef_ldl * row_sample["num__ldl"].values 
 + coef_famhist * row_sample["cat__famhist"].values 
)


def sigmoid(z):
    return 1/(1 + np.exp(-z))

sigmoid(pred_val)

array([0.37400428])

In [17]:
print(r"x_{i, \text{obesity}}", round(row_sample["num__obesity"].values[0], 2))
print(r"x_{i, \text{alcohol}}", round(row_sample["num__alcohol"].values[0], 2))
print(r"x_{i, \text{typea}}", round(row_sample["num__typea"].values[0], 2))
print(r"x_{i, \text{tobacco}}", round(row_sample["num__tobacco"].values[0], 2))
print(r"x_{i, \text{sbp}}", round(row_sample["num__sbp"].values[0], 2))
print(r"x_{i, \text{age}}", round(row_sample["num__age"].values[0], 2))
print(r"x_{i, \text{ldl}}", round(row_sample["num__ldl"].values[0], 2))
print(r"x_{i, \text{famhist}}", round(row_sample["cat__famhist"].values[0], 2))

x_{i, \text{obesity}} 0.67
x_{i, \text{alcohol}} 0.27
x_{i, \text{typea}} 0.19
x_{i, \text{tobacco}} -0.49
x_{i, \text{sbp}} 0.28
x_{i, \text{age}} 1.38
x_{i, \text{ldl}} -0.16
x_{i, \text{famhist}} 0.0


In [15]:
pred_val

array([-0.51507631])

$$
\begin{align*}
\text{weights} & \qquad \text{observation} \\
w_\text{intercept} = -1.10 &\\
w_{\text{obesity}} = -0.16  &\qquad  x_{i, \text{obesity}} = 0.67 \\
w_{\text{alcohol}} = -0.052  &\qquad  x_{i, \text{alcohol}} = 0.27 \\
w_{\text{typea}} = 0.37  &\qquad  x_{i, \text{typea}} = 0.19 \\
w_{\text{tobacco}} = 0.41  &\qquad  x_{i, \text{tobacco}} = -0.49 \\
w_{\text{sbp}} = 0.16  &\qquad  x_{i, \text{sbp}} = 0.28 \\
w_{\text{age}} = 0.75  &\qquad  x_{i, \text{age}} = 1.38 \\
w_{\text{ldl}} = 0.34  &\qquad  x_{i, \text{ldl}} = -0.16 \\
w_{\text{famhist}} = 0.89  &\qquad x_{i, \text{famhist}}=     0 \\
\end{align*}
$$

$$
\begin{align*}
y_i &= \sigma (w_{\text{intercept}} 
+ w_{\text{obesity}} \cdot x_{i, \text{obesity}}\\
&\quad 
+ w_{\text{alcohol}} \cdot x_{i, \text{alcohol}}
+ w_{\text{typea}} \cdot x_{i, \text{typea}}\\
&\quad 
+ w_{\text{tobacco}} \cdot x_{i, \text{tobacco}} 
+ w_{\text{sbp}} \cdot x_{i, \text{sbp}}\\
&\quad 
+ w_{\text{age}} \cdot x_{i, \text{age}}  
+ w_{\text{ldl}} \cdot x_{i, \text{ldl}} \\
&\quad 
+ w_{\text{famhist}} \cdot x_{i, \text{famhist}})\\
&= \sigma(-1.29
+  (-0.16) \cdot  0.67 \\
&\quad 
+  (-0.052) \cdot  0.27 
+  0.37 \cdot  0.19  \\
&\quad 
+  0.41 \cdot (-0.49)
+  0.16 \cdot  0.28 \\
&\quad  
+  0.75 \cdot  1.38 
+  0.34 \cdot (-0.16)\\
&\quad 
+  0.89 \cdot    0  ) \\
&= \frac{1}{1+\exp^{-0.5151}} \\
&= 0.374
\end{align*} 
$$ 

In [21]:
y[1:2]

,chd
1,1
